#### Env setup (uv)

1. Install Google Cloud SDK
   
2. Create the environment:
   ```bash
   uv venv .poc-lora
   ```

3. Activate it (terminal step; run before continuing):
   ```bash
   source .poc-lora/bin/activate
   ```

4. Install notebook + model dependencies:
   ```bash
   uv pip install jupyter ipykernel "transformers>=4.41" accelerate safetensors pillow pandas datasets gcsfs
   ```

5. Register the kernel:
   ```bash
   python -m ipykernel install --user --name=poc-lora --display-name="POC LoRA"
   ```

6. In `experiments/poc.ipynb`, pick the `Where You LoRA (vision)` kernel before running any cells.

Imports

In [ ]:
!pip install -U "transformers>=4.45.0" accelerate bitsandbytes "torchvision" pillow datasets \
               gcsfs google-cloud-storage google-auth google-colab rich

INFO: pip is looking at multiple versions of fsspec[http] to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 148.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 156.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB

In [ ]:

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from transformers import AutoProcessor, LlavaOnevisionForConditionalGeneration
import pandas as pd
from datasets import load_dataset

import torch
import gc

gc.collect()
torch.cuda.empty_cache()

In [ ]:
#del out, H, V, inp
import gc, torch
gc.collect()
torch.cuda.empty_cache()

Constants

In [ ]:
target_modules = [
    "q_proj", "k_proj", "v_proj", "o_proj",  # attention
    "up_proj", "gate_proj", "down_proj",    # MLP
    "vision_resampler.dense",              # vision adapter
]

Dataset

In [ ]:
from datasets import load_dataset
ds = load_dataset("merve/vqav2-small", split="validation")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/403 [00:00<?, ?B/s]

data/validation-00000-of-00007.parquet:   0%|          | 0.00/484M [00:00<?, ?B/s]

data/validation-00001-of-00007.parquet:   0%|          | 0.00/486M [00:00<?, ?B/s]

data/validation-00002-of-00007.parquet:   0%|          | 0.00/483M [00:00<?, ?B/s]

data/validation-00003-of-00007.parquet:   0%|          | 0.00/486M [00:00<?, ?B/s]

data/validation-00004-of-00007.parquet:   0%|          | 0.00/475M [00:00<?, ?B/s]

data/validation-00005-of-00007.parquet:   0%|          | 0.00/484M [00:00<?, ?B/s]

data/validation-00006-of-00007.parquet:   0%|          | 0.00/479M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/21435 [00:00<?, ? examples/s]

In [ ]:
ds.shape

(21435, 3)

In [ ]:
test_batch = mini[0:1000]

Model

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForCausalLM
from PIL import Image

MODEL_ID = "lmms-lab/LLaVA-OneVision-1.5-8B-Instruct"

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, trust_remote_code=True, torch_dtype="auto", device_map="auto"
).eval()
if hasattr(processor, "tokenizer"):
    processor.tokenizer.padding_side = "left"

preprocessor_config.json:   0%|          | 0.00/573 [00:00<?, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/2.17G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/225 [00:00<?, ?B/s]

In [ ]:
import os, random, json, math
import torch
import torch.nn.functional as Fnn
from datasets import load_dataset
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM

In [ ]:
#chat template
img = Image.open("/content/test_image.jpg").convert("RGB")
conversation = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": "What is in the picture?"},
        ],
    }
]

In [ ]:
prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)

Metrics Inspection

1. Access to Embeddings we need to calculate metrics

What do we need to extract (4 taps)
1. Vision Tower features pre projection stage
2. Vproj (post projection visual tokens)
3. Text tokens
4. fused/text-decoder hidden states

In [ ]:
inputs = processor(images=img, text=prompt, return_tensors="pt", padding=True)

inputs = {k: (v.to(model.device) if isinstance(v, torch.Tensor) else v) for k, v in inputs.items()}

NameError: name 'img' is not defined

In [ ]:
with torch.no_grad():
    out = model(**inputs, output_hidden_states=True, return_dict=True)

The final decoder hidden states, fused reps

In [ ]:
Fused = out.hidden_states[-1]

Projected image tokens after the connector. These are the visual embeddings that get fed into the LLM; use them as Vproj in your analyses.

In [ ]:
with torch.no_grad():
    V = model.visual(
        inputs["pixel_values"].to(next(model.parameters()).device,
                                  dtype=next(model.parameters()).dtype),
        grid_thw=inputs["image_grid_thw"].to(next(model.parameters()).device),
    )

The count of visual tokens that precede the text tokens in the fused sequence. In your code you estimated it from V.shape[1] and assumed those appear first in F—that’s the usual layout in LLaVA-style models and matches the docs

In [ ]:
num_img_tokens = V.shape[-2] if V.dim() == 3 else V.shape[0]

In [ ]:
B, S, _ = Fused.shape

Boolean masks over the fused sequence (F) that let you select just the text positions or just the image positions.

In [ ]:
image_mask = torch.zeros(B, S, dtype=torch.bool, device=Fused.device)
image_mask[:, :num_img_tokens] = True

In [ ]:
text_mask = ~image_mask

How to return into embeddings

In [ ]:
import torch
import torch.nn.functional as Fnn   # <- avoid 'F' to prevent collisions
from typing import Optional,Dict

def mean_pool(x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
    if x.dim() == 2:
        x = x.unsqueeze(0)
    if mask is None:
        return x.mean(dim=1)
    w = mask.float().unsqueeze(-1)
    return (x * w).sum(dim=1) / w.sum(dim=1).clamp_min(1e-9)

In [ ]:
# Visual (post-projector)
Vproj_vec = mean_pool(V)

# Text embeddings AFTER fusion (optional diagnostic)
T_fused_vec = mean_pool(Fused, text_mask)

# Whole fused sequence (sometimes handy)
F_vec = mean_pool(Fused)

In [ ]:
# Sanity: norms shouldn't be ~0 or NaN
print("||Vproj_vec|| min/max:", Vproj_vec.norm(dim=-1).min().item(),
      Vproj_vec.norm(dim=-1).max().item())
print("||T_fused_vec|| min/max:", T_fused_vec.norm(dim=-1).min().item(),
      T_fused_vec.norm(dim=-1).max().item())

||Vproj_vec|| min/max: 52.75 52.75
||T_fused_vec|| min/max: 83.47162628173828 83.47162628173828


Metrics

In [ ]:
import torch
import torch.nn.functional as F

EPS = 1e-8

def l2norm(x):
    return x / (x.norm(dim=-1, keepdim=True).clamp_min(EPS))

def intra_modal_similarity(X):
    Xn = l2norm(X)
    S = Xn @ Xn.T                                    # [N,N]
    n = S.size(0)
    M = ~torch.eye(n, dtype=bool, device=S.device)
    vals = S[M]
    vals = vals[torch.isfinite(vals)]
    return vals.mean().item() if vals.numel()>0 else 0.0

def modality_gap(I, T):
    return (I.mean(0) - T.mean(0)).norm().item()

def intermodal_metrics(I, T):
    In, Tn = l2norm(I), l2norm(T)
    pos = (In * Tn).sum(-1)                          # [N]
    # sample negatives
    perm = torch.randperm(I.size(0), device=I.device)
    neg = (In * Tn[perm]).sum(-1)
    # remove any NaNs/Infs
    pos = pos[torch.isfinite(pos)]
    neg = neg[torch.isfinite(neg)]
    if pos.numel()==0 or neg.numel()==0:
        return {"pos_mean": 0.0, "neg_mean": 0.0, "margin": 0.0, "auc_proxy": 0.0}
    margin = (pos.mean() - neg.mean()).item()
    auc_proxy = (pos[:,None] > neg[None,:]).float().mean().item()
    return {"pos_mean": pos.mean().item(), "neg_mean": neg.mean().item(),
            "margin": margin, "auc_proxy": auc_proxy}

def effective_rank(Z, center=True):
    Zc = Z - Z.mean(0, keepdim=True) if center else Z
    S = torch.linalg.svd(Zc.to(torch.float64), full_matrices=False).S
    ssum = S.sum()
    if S.numel()==0 or ssum<=EPS:
        return 0.0
    p = (S / ssum).clamp_min(EPS)
    H = -(p * p.log()).sum()
    return torch.exp(H).item()

def concentration_ratio(Z, k=1, center=True):
    Zc = Z - Z.mean(0, keepdim=True) if center else Z
    S = torch.linalg.svd(Zc.to(torch.float64), full_matrices=False).S
    ssum = S.sum()
    if S.numel()==0 or ssum<=EPS:
        return 0.0
    k = min(k, S.numel())
    return (S[:k].sum() / ssum).item()

def linear_cka(X, Y, center=True):
    Xc = X - X.mean(0, keepdim=True) if center else X
    Yc = Y - Y.mean(0, keepdim=True) if center else Y
    K, L = Xc @ Xc.T, Yc @ Yc.T
    num = (K*L).sum()
    den = torch.sqrt((K*K).sum()) * torch.sqrt((L*L).sum())
    return (num / den).item() if den > EPS else 0.0

def compute_all_metrics_from_your_vars(Vproj_vec, T_fused_vec):
    out = {}
    out["intra_sim_img"] = intra_modal_similarity(Vproj_vec)
    out["intra_sim_txt"] = intra_modal_similarity(T_fused_vec)
    out["modality_gap"]  = modality_gap(Vproj_vec, T_fused_vec)
    out.update(intermodal_metrics(Vproj_vec, T_fused_vec))
    out["erank_img"] = effective_rank(Vproj_vec)
    out["erank_txt"] = effective_rank(T_fused_vec)
    out["cr1_img"]   = concentration_ratio(Vproj_vec, k=1)
    out["cr1_txt"]   = concentration_ratio(T_fused_vec, k=1)
    out["cka_img_txt"] = linear_cka(Vproj_vec, T_fused_vec)
    return out

In [ ]:
metrics = compute_all_metrics_from_your_vars(Vproj_vec=V, T_fused_vec= Fused)  # or pass V0_vec if you have it
metrics

{'intra_sim_img': nan,
 'intra_sim_txt': nan,
 'modality_gap': 97.76068115234375,
 'inter_pos_mean': 0.022964291274547577,
 'inter_neg_mean': 0.022964291274547577,
 'inter_margin': 0.0,
 'inter_auc_proxy': 0.0,
 'erank_img': nan,
 'erank_txt': nan,
 'cr1_img': nan,
 'cr1_txt': nan,
 'cka_img_txt': nan}

Generate answer

In [ ]:
img = Image.open("/content/test_image.jpg").convert("RGB")
conversation = [{
    "role": "user",
    "content": [
        {"type": "image", "image": img},
        {"type": "text", "text": "Are the bananas blue?"}
    ],
}]


# 1) Proper multimodal prompt (required for OneVision)
inputs = processor.apply_chat_template(
    conversation, add_generation_prompt=True, tokenize=True,
    return_tensors="pt", return_dict=True
)  # HF multimodal chat templating. :contentReference[oaicite:2]{index=2}

# 2) Align devices (DON'T change integer dtypes)
def align_for_generate(model, inputs):
    # text indices/masks -> device of input embedding weights
    emb_dev = model.get_input_embeddings().weight.device
    for k in ("input_ids", "attention_mask", "position_ids"):
        if k in inputs and isinstance(inputs[k], torch.Tensor):
            inputs[k] = inputs[k].to(device=emb_dev)  # keep Long/Bool
    # image tensors -> device/dtype of model params (works for OneVision 1.5)
    first_param = next(model.parameters())
    if "pixel_values" in inputs:
        inputs["pixel_values"] = inputs["pixel_values"].to(first_param.device, dtype=first_param.dtype)
    if "image_grid_thw" in inputs:
        inputs["image_grid_thw"] = inputs["image_grid_thw"].to(first_param.device)
    return inputs

inputs = align_for_generate(model, inputs)

# 3) Generate (choose greedy OR sampling—don’t mix)
with torch.no_grad():
    gen_ids = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,          # greedy => temperature/top_p are ignored
        eos_token_id=processor.tokenizer.eos_token_id,
        pad_token_id=processor.tokenizer.eos_token_id,
    )
    # If you want sampling instead, use:
    # gen_ids = model.generate(**inputs, max_new_tokens=128, do_sample=True, temperature=0.2, top_p=0.9)

# 4) Decode only the new tokens
start = inputs["input_ids"].shape[1]
answer = processor.tokenizer.decode(gen_ids[0][start:], skip_special_tokens=True).strip()
print(answer)

yes


Lets test a batch

In [ ]:

import os
import torch

MODEL_ID     = "lmms-lab/LLaVA-OneVision-1.5-8B-Instruct"
N_SAMPLES    = 1000
SEED         = 42
BATCH        = 3          # lower if you OOM
MAX_NEW_TOKS = 16
DO_FUNCTIONAL= True        # set False to skip Δ_img/Δ_txt generation
OUT_DIR      = "./llava_1k_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------- METRICS ----------------
EPS = 1e-8

def l2norm(x): return x / (x.norm(dim=-1, keepdim=True).clamp_min(EPS))

import torch
EPS = 1e-8

def mean_pool(x: torch.Tensor, mask: torch.Tensor | None = None, fallback: str = "mean") -> torch.Tensor:
    """
    Safe mean-pool over sequence with dtype/device guards.
    x: [B,T,D] or [T,D]; mask: [B,T] bool
    """
    if x.dim() == 2:
        x = x.unsqueeze(0)

    d   = x.dtype
    dev = x.device

    if mask is None:
        return x.mean(1)

    w = mask.to(dev).float().unsqueeze(-1)           # [B,T,1], float32
    denom  = w.sum(1).clamp_min(1.0).to(d)           # [B,1], cast to x.dtype
    pooled = ((x * w.to(d)).sum(1) / denom)          # stays in x.dtype

    if fallback == "mean":
        no_tokens = (w.sum(1).squeeze(-1) < 0.5)     # [B] bool
        if no_tokens.any():
            rhs = x[no_tokens].mean(1)               # dtype = x.dtype
            pooled[no_tokens] = rhs.to(pooled.dtype) # explicit match
    return pooled

def intra_modal_similarity(X):
    Xn = l2norm(X); S = Xn @ Xn.T
    n = S.size(0); M = ~torch.eye(n, dtype=bool, device=S.device)
    vals = S[M]; vals = vals[torch.isfinite(vals)]
    return vals.mean().item() if vals.numel()>0 else 0.0

def modality_gap(I, T): return (I.mean(0) - T.mean(0)).norm().item()

def intermodal_metrics(I, T):
    In, Tn = l2norm(I), l2norm(T)
    pos = (In*Tn).sum(-1)
    perm = torch.randperm(I.size(0), device=I.device)
    neg = (In*Tn[perm]).sum(-1)
    pos = pos[torch.isfinite(pos)]; neg = neg[torch.isfinite(neg)]
    if pos.numel()==0 or neg.numel()==0:
        return {"pos_mean":0.0,"neg_mean":0.0,"margin":0.0,"auc_proxy":0.0}
    margin = (pos.mean()-neg.mean()).item()
    auc = (pos[:,None] > neg[None,:]).float().mean().item()
    return {"pos_mean": pos.mean().item(), "neg_mean": neg.mean().item(), "margin": margin, "auc_proxy": auc}

def effective_rank(Z, center=True):
    Zc = Z - Z.mean(0, keepdim=True) if center else Z
    S = torch.linalg.svd(Zc.to(torch.float64), full_matrices=False).S
    ssum = S.sum()
    if S.numel()==0 or ssum<=EPS: return 0.0
    p = (S/ssum).clamp_min(EPS)
    H = -(p*p.log()).sum()
    return torch.exp(H).item()

def concentration_ratio(Z, k=1, center=True):
    Zc = Z - Z.mean(0, keepdim=True) if center else Z
    S = torch.linalg.svd(Zc.to(torch.float64), full_matrices=False).S
    ssum = S.sum()
    if S.numel()==0 or ssum<=EPS: return 0.0
    k = min(k, S.numel())
    return (S[:k].sum()/ssum).item()

def linear_cka(X, Y, center=True):
    Xc = X - X.mean(0, keepdim=True) if center else X
    Yc = Y - Y.mean(0, keepdim=True) if center else Y
    K, L = Xc @ Xc.T, Yc @ Yc.T
    num = (K*L).sum()
    den = torch.sqrt((K*K).sum()) * torch.sqrt((L*L).sum())
    return (num/den).item() if den > EPS else 0.0

def compute_all_metrics_from_vectors(Vproj_vec, T_vec):
    out = {}
    out["intra_sim_img"] = intra_modal_similarity(Vproj_vec)
    out["intra_sim_txt"] = intra_modal_similarity(T_vec)
    out["modality_gap"]  = modality_gap(Vproj_vec, T_vec)
    out.update(intermodal_metrics(Vproj_vec, T_vec))
    out["erank_img"] = effective_rank(Vproj_vec)
    out["erank_txt"] = effective_rank(T_vec)
    out["cr1_img"]   = concentration_ratio(Vproj_vec, k=1)
    out["cr1_txt"]   = concentration_ratio(T_vec, k=1)
    out["cka_img_txt"] = linear_cka(Vproj_vec, T_vec)
    return out

In [ ]:
idxs = list(range(len(ds))); random.Random(SEED).shuffle(idxs)
idxs = idxs[:N_SAMPLES]
ds_small = ds.select(idxs)

In [ ]:
img_col = "image" if "image" in ds_small.column_names else "image_path"
q_col   = "question"
ans_col = "answers" if "answers" in ds_small.column_names else ("multiple_choice_answer" if "multiple_choice_answer" in ds_small.column_names else None)

In [ ]:
# ---------------- HELPERS ----------------
def align_for_model(inp):
    # move only necessary tensors; keep integer dtypes intact
    emb_dev = model.get_input_embeddings().weight.device
    for k in ("input_ids","attention_mask","position_ids"):
        if k in inp and isinstance(inp[k], torch.Tensor):
            inp[k] = inp[k].to(emb_dev)
    first_param = next(model.parameters())
    if "pixel_values" in inp:
        inp["pixel_values"] = inp["pixel_values"].to(first_param.device, dtype=first_param.dtype)
    if "image_grid_thw" in inp:
        inp["image_grid_thw"] = inp["image_grid_thw"].to(first_param.device)
    return inp

def quick_answer(conv, do_sample=False):
    inp = processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=True,
                                        return_tensors="pt", return_dict=True)
    inp = align_for_model(inp)
    with torch.no_grad():
        gen = model.generate(
            **inp,
            max_new_tokens=MAX_NEW_TOKS,
            do_sample=do_sample,            # sampling only if True (otherwise temperature/top_p are ignored)
            temperature=(0.2 if do_sample else None),
            top_p=(0.9 if do_sample else None),
            eos_token_id=processor.tokenizer.eos_token_id,
            pad_token_id=processor.tokenizer.eos_token_id,
        )
    start = inp["input_ids"].shape[1]
    return processor.tokenizer.decode(gen[0][start:], skip_special_tokens=True).strip()

def light_match(pred, gold):
    s = str(pred).lower().strip().replace(".", "")
    if isinstance(gold, list):
        gold_strs = []
        for g in gold:
            if isinstance(g, dict) and "answer" in g: gold_strs.append(str(g["answer"]).lower().strip())
            else: gold_strs.append(str(g).lower().strip())
        return int(any(gs in s or s in gs for gs in gold_strs))
    elif gold is None:
        return 0
    else:
        g = str(gold).lower().strip().replace(".", "")
        return int(g in s or s in g)

In [ ]:
# --- FIXED main loop ---
from datasets import Image as HFImage
from PIL import Image

# ensure PIL decoded images
ds_small = ds_small.cast_column(img_col, HFImage(decode=True))

all_V, all_T, rows = [], [], []
acc_full = acc_txt = acc_img = 0

N = len(ds_small)
for start in range(0, N, BATCH):
    end = min(start + BATCH, N)
    batch_idx = list(range(start, end))
    batch_ds  = ds_small.select(batch_idx)          # row-wise mini-dataset

    # 1) Build batched conversations for embeddings (not generation)
    convs = []
    imgs_for_gen = []                                # keep PILs for later generate calls
    for i in range(len(batch_ds)):
        ex  = batch_ds[i]                            # row dict
        img = ex[img_col]
        if not isinstance(img, Image.Image):
            img = Image.open(img).convert("RGB")
        q = ex[q_col]
        convs.append({
            "role": "user",
            "content": [
                {"type": "image", "image": img},
                {"type": "text",  "text": q},
            ],
        })
        imgs_for_gen.append(img)

    # 2) Forward pass to get fused states H and visual tokens V
    inp = processor.apply_chat_template(
        convs, add_generation_prompt=False, tokenize=True,
        return_tensors="pt", return_dict=True
    )
    inp = align_for_model(inp)
    with torch.no_grad():
        out = model(**inp, output_hidden_states=True, return_dict=True)
        H   = out.hidden_states[-1]                               # [B, T_all, D]
        V   = model.visual(inp["pixel_values"], grid_thw=inp["image_grid_thw"])  # [B, T_img, D]

    # 3) Build masks (image tokens come FIRST in H for OV 1.5)
    B = H.shape[0]; S = H.shape[1]; Ti = V.shape[1]
    image_mask = torch.zeros(B, S, dtype=torch.bool, device=H.device)
    image_mask[:, :Ti] = True
    text_mask = ~image_mask

    # 4) Pool to per-sample vectors
    Vproj_vec   = mean_pool(V)                        # [B, D]  (post-projector visual)
    T_fused_vec = mean_pool(H, mask=text_mask)        # [B, D]  (text tokens AFTER fusion)

    all_V.append(Vproj_vec.cpu()); all_T.append(T_fused_vec.cpu())

    # 5) (Optional) generation for functional metrics, now using row dicts
    if DO_FUNCTIONAL:
        # full
        full_preds = []
        for i in range(len(batch_ds)):
            ex  = batch_ds[i]
            img = imgs_for_gen[i]
            conv = [{"role":"user","content":[
                        {"type":"image","image":img},
                        {"type":"text","text":ex[q_col]}
                    ]}]
            full_preds.append(quick_answer(conv, do_sample=False))

        # text-only (mask image)
        txt_preds = []
        for i in range(len(batch_ds)):
            ex = batch_ds[i]
            conv = [{"role":"user","content":[{"type":"text","text":ex[q_col]}]}]
            txt_preds.append(quick_answer(conv, do_sample=False))

        # image-only (empty text)
        img_preds = []
        for i in range(len(batch_ds)):
            ex  = batch_ds[i]
            img = imgs_for_gen[i]
            conv = [{"role":"user","content":[
                        {"type":"image","image":img},
                        {"type":"text","text":""}
                    ]}]
            img_preds.append(quick_answer(conv, do_sample=False))

        # accumulate accuracy + keep per-row
        for i in range(len(batch_ds)):
            ex   = batch_ds[i]
            gold = ex.get(ans_col, None)
            acc_full += light_match(full_preds[i], gold)
            acc_txt  += light_match(txt_preds[i],  gold)
            acc_img  += light_match(img_preds[i],  gold)
            rows.append({
                "question": ex[q_col],
                "gold": gold,
                "pred_full": full_preds[i],
                "pred_text_only": txt_preds[i],
                "pred_image_only": img_preds[i],
            })


In [ ]:
 # stack pooled vectors and save
V_all = torch.cat(all_V, dim=0)   # [N, D]
T_all = torch.cat(all_T, dim=0)   # [N, D]
torch.save({"Vproj_vec": V_all, "T_fused_vec": T_all}, os.path.join(OUT_DIR, "vectors.pt"))

# compute metrics
metrics = compute_all_metrics_from_vectors(V_all, T_all)
print("\n=== Representation/Distribution Metrics (N≈1k) ===")
for k,v in metrics.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

# functional deltas
if DO_FUNCTIONAL and ans_col is not None:
    total = len(ds_small)
    M_full = acc_full/total
    M_txt  = acc_txt/total
    M_img  = acc_img/total
    print(f"\n=== Functional Metrics over {total} items ===")
    print(f"Acc_full     : {M_full:.4f}")
    print(f"Acc_textOnly : {M_txt:.4f}")
    print(f"Acc_imgOnly  : {M_img:.4f}")
    print(f"Δ_img = Acc_full - Acc_textOnly = {(M_full - M_txt):.4f}")
    print(f"Δ_txt = Acc_full - Acc_imgOnly  = {(M_full - M_img):.4f}")
    with open(os.path.join(OUT_DIR, "preds.jsonl"), "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    with open(os.path.join(OUT_DIR, "functional.json"), "w", encoding="utf-8") as f:
        json.dump({
            "Acc_full": M_full, "Acc_textOnly": M_txt, "Acc_imgOnly": M_img,
            "Delta_img": M_full - M_txt, "Delta_txt": M_full - M_img
        }, f, indent=2)
print(f"\nSaved vectors to {os.path.join(OUT_DIR, 'vectors.pt')}")
if DO_FUNCTIONAL: print(f"Saved predictions/metrics to {OUT_DIR}")


=== Representation/Distribution Metrics (N≈1k) ===
intra_sim_img: 0.9453
intra_sim_txt: 0.8672
modality_gap: 87.5000
pos_mean: 0.0542
neg_mean: 0.0447
margin: 0.0095
auc_proxy: 0.8062
erank_img: 66.3004
erank_txt: 78.4232
cr1_img: 0.0739
cr1_txt: 0.0527
cka_img_txt: 0.7266

=== Functional Metrics over 100 items ===
Acc_full     : 0.7500
Acc_textOnly : 0.2300
Acc_imgOnly  : 0.1600
Δ_img = Acc_full - Acc_textOnly = 0.5200
Δ_txt = Acc_full - Acc_imgOnly  = 0.5900

Saved vectors to ./llava_1k_outputs/vectors.pt
Saved predictions/metrics to ./llava_1k_outputs


In [ ]:
# --- FIXED main loop ---
from datasets import Image as HFImage
from PIL import Image

# ensure PIL decoded images
ds_small = ds_small.cast_column(img_col, HFImage(decode=True))

all_V, all_T, rows = [], [], []
acc_full = acc_txt = acc_img = 0

N = len(ds_small)
for start in range(0, N, BATCH):
    end = min(start + BATCH, N)
    batch_idx = list(range(start, end))
    batch_ds  = ds_small.select(batch_idx)          # row-wise mini-dataset

    # 1) Build batched conversations for embeddings (not generation)
    convs = []
    imgs_for_gen = []                                # keep PILs for later generate calls
    for i in range(len(batch_ds)):
        ex  = batch_ds[i]                            # row dict
        img = ex[img_col]
        if not isinstance(img, Image.Image):
            img = Image.open(img).convert("RGB")
        q = ex[q_col]
        convs.append({
            "role": "user",
            "content": [
                {"type": "image", "image": img},
                {"type": "text",  "text": q},
            ],
        })
        imgs_for_gen.append(img)

    # 2) Forward pass to get fused states H and visual tokens V
    inp = processor.apply_chat_template(
        convs, add_generation_prompt=False, tokenize=True,
        return_tensors="pt", return_dict=True
    )
    inp = align_for_model(inp)
    with torch.no_grad():
        out = model(**inp, output_hidden_states=True, return_dict=True)
        H   = out.hidden_states[-1]                               # [B, T_all, D]
        V   = model.visual(inp["pixel_values"], grid_thw=inp["image_grid_thw"])  # [B, T_img, D]

    # 3) Build masks (image tokens come FIRST in H for OV 1.5)
    B = H.shape[0]; S = H.shape[1]; Ti = V.shape[1]
    image_mask = torch.zeros(B, S, dtype=torch.bool, device=H.device)
    image_mask[:, :Ti] = True
    text_mask = ~image_mask

    # 4) Pool to per-sample vectors
    Vproj_vec   = mean_pool(V)                        # [B, D]  (post-projector visual)
    T_fused_vec = mean_pool(H, mask=text_mask)        # [B, D]  (text tokens AFTER fusion)

    all_V.append(Vproj_vec.cpu()); all_T.append(T_fused_vec.cpu())

    # 5) (Optional) generation for functional metrics, now using row dicts
    if DO_FUNCTIONAL:
        # full
        full_preds = []
        for i in range(len(batch_ds)):
            ex  = batch_ds[i]
            img = imgs_for_gen[i]
            conv = [{"role":"user","content":[
                        {"type":"image","image":img},
                        {"type":"text","text":ex[q_col]}
                    ]}]
            full_preds.append(quick_answer(conv, do_sample=False))

        # text-only (mask image)
        txt_preds = []
        for i in range(len(batch_ds)):
            ex = batch_ds[i]
            conv = [{"role":"user","content":[{"type":"text","text":ex[q_col]}]}]
            txt_preds.append(quick_answer(conv, do_sample=False))

        # image-only (empty text)
        img_preds = []
        for i in range(len(batch_ds)):
            ex  = batch_ds[i]
            img = imgs_for_gen[i]
            conv = [{"role":"user","content":[
                        {"type":"image","image":img},
                        {"type":"text","text":""}
                    ]}]
            img_preds.append(quick_answer(conv, do_sample=False))

        # accumulate accuracy + keep per-row
        for i in range(len(batch_ds)):
            ex   = batch_ds[i]
            gold = ex.get(ans_col, None)
            acc_full += light_match(full_preds[i], gold)
            acc_txt  += light_match(txt_preds[i],  gold)
            acc_img  += light_match(img_preds[i],  gold)
            rows.append({
                "question": ex[q_col],
                "gold": gold,
                "pred_full": full_preds[i],
                "pred_text_only": txt_preds[i],
                "pred_image_only": img_preds[i],
            })


In [ ]:
 # stack pooled vectors and save
V_all = torch.cat(all_V, dim=0)   # [N, D]
T_all = torch.cat(all_T, dim=0)   # [N, D]
torch.save({"Vproj_vec": V_all, "T_fused_vec": T_all}, os.path.join(OUT_DIR, "vectors.pt"))

# compute metrics
metrics = compute_all_metrics_from_vectors(V_all, T_all)
print("\n=== Representation/Distribution Metrics (N≈1k) ===")
for k,v in metrics.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

# functional deltas
if DO_FUNCTIONAL and ans_col is not None:
    total = len(ds_small)
    M_full = acc_full/total
    M_txt  = acc_txt/total
    M_img  = acc_img/total
    print(f"\n=== Functional Metrics over {total} items ===")
    print(f"Acc_full     : {M_full:.4f}")
    print(f"Acc_textOnly : {M_txt:.4f}")
    print(f"Acc_imgOnly  : {M_img:.4f}")
    print(f"Δ_img = Acc_full - Acc_textOnly = {(M_full - M_txt):.4f}")
    print(f"Δ_txt = Acc_full - Acc_imgOnly  = {(M_full - M_img):.4f}")
    with open(os.path.join(OUT_DIR, "preds.jsonl"), "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    with open(os.path.join(OUT_DIR, "functional.json"), "w", encoding="utf-8") as f:
        json.dump({
            "Acc_full": M_full, "Acc_textOnly": M_txt, "Acc_imgOnly": M_img,
            "Delta_img": M_full - M_txt, "Delta_txt": M_full - M_img
        }, f, indent=2)
print(f"\nSaved vectors to {os.path.join(OUT_DIR, 'vectors.pt')}")
if DO_FUNCTIONAL: print(f"Saved predictions/metrics to {OUT_DIR}")


=== Representation/Distribution Metrics (N≈1k) ===
intra_sim_img: 0.9805
intra_sim_txt: 0.9375
modality_gap: 88.5000
pos_mean: 0.0515
neg_mean: 0.0481
margin: 0.0034
auc_proxy: 0.7091
erank_img: 146.1958
erank_txt: 228.7270
cr1_img: 0.0543
cr1_txt: 0.0383
cka_img_txt: 0.6719

=== Functional Metrics over 1000 items ===
Acc_full     : 0.7220
Acc_textOnly : 0.1950
Acc_imgOnly  : 0.1300
Δ_img = Acc_full - Acc_textOnly = 0.5270
Δ_txt = Acc_full - Acc_imgOnly  = 0.5920

Saved vectors to ./llava_1k_outputs/vectors.pt
Saved predictions/metrics to ./llava_1k_outputs


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

LoRA Inspection